# Flood Mapping with EODC Dask Gateway

Welcome to this workshop!

Here we intend to show the steps to create Floods Map with Sentinel-1 radar images. We replicate in this package the work of Bauer-Marschallinger et al. (2022) on the TU Wien Bayesian-based flood mapping algorithm. 
The computation is carried out remotely via EODC Dask Gateway and the data are accessed via STAC with odc-stac.



## 1. Setting up the connection to EODC Dask Gateway
Authentication is required through a username and password. It is the same used in the Jupyterhub login.

In [1]:
from eodc_connect.dask import EODCDaskGateway
from rich.console import Console
from rich.prompt import Prompt

console = Console()
your_username = Prompt.ask(prompt="Enter your Username")
gateway = EODCDaskGateway(username=your_username)

Enter your Username:

 workshop_test
Enter your password: ········


In [49]:
from collections.abc import Sequence

import hvplot.xarray  # noqa
import numpy as np
import pystac_client
import xarray as xr
from dask.distributed import Client, wait  # noqa: F401

from odc import stac as odc_stac
from pystac_client.item_search import ItemCollection

Now the cluster configuration can be defined. Then, we spawn and scale a new cluster.

In [25]:
# Cluster configuration
cluster_options = gateway.cluster_options()
cluster_options.image = "ghcr.io/eodcgmbh/cluster_image:2025.9.3"
cluster_options.worker_cores = 4
cluster_options.worker_memory = 8.0

# Spawn a new cluster
cluster = gateway.new_cluster(cluster_options)

# Specify the number of workers
cluster.scale(1)

# Get the client
client = cluster.get_client()
cluster

In [50]:
# If you already created a cluster uncomment the following:

clusters = gateway.list_clusters()
cluster_name = clusters[0].name

cluster = gateway.connect(cluster_name)
cluster

Accessing the Dask dashboard for monitoring the cluster.

In [5]:
cluster.dashboard_link

'https://dask.services.eodc.eu/clusters/dask-gateway.bde24df9f234496ea42e87dd02ac0efa/status'

Setting up the coordinate reference system, resolution and size of chunks for Dask to optimize the performance.

In [51]:
# Coordinate Reference System - World Geodetic System 1984 (WGS84) in this case
crs = "EPSG:4326"

# Resolution
res = 0.00018

# Size of data chunks
chunks = {"time": 1, "latitude": 1300, "longitude": 1300}

## 2. Use case: Northern Germany Flood

Storm Babet hit the Danish and Northern German coast on the 20th of October 2023 [Wikipedia](https://en.wikipedia.org/wiki/Storm_Babet). 

An area around Zingst at the Baltic coast of Northern Germany is selected as the study area.

In [52]:
# Time range of the event
time_range = "2023-10-11/2023-10-25"

# Definition of the area trough latitude and longitude
minlon, maxlon = 12.3, 13.1
minlat, maxlat = 54.3, 54.6
bounding_box = (minlon, minlat, maxlon, maxlat)

## 3. Getting the data: EODC STAC Catalog

The data is obtained through the EODC STAC Catalog. The connection to it is made via `pystac_client`.

We will get and process the following data:

* 3.1 [Microwave Backscatter Measurements](#31-microwave-backscatter-measurements)
* 3.2 [Harmonic Parameters](#32-harmonic-parameters)
* 3.3 [Local Incidence Angles](#33-local-incidence-angles-lia) 
* 3.4 [GFM World Cover](#34-gfm-water-mask)


In [53]:
# Create the catalog object from EODC
eodc_catalog = pystac_client.Client.open("https://stac.eodc.eu/api/v1")

# Get all collections
collections = eodc_catalog.get_collections()

### 3.1 Microwave Backscatter Measurements

The characteristics of backscattering over land and water differ considerably. With this knowledge we can detect whenever a pixel with a predominant land like signature changes to a water like signature in the event of flooding.

<figure style="text-align: center;">
  <img src="https://www.gsi.ie/images/images/SAR_mapping_land_water.jpg" alt="SAR Map" width="400">
  <figcaption> *Schematic backscattering over land and water. Image from [Geological Survey Ireland](https://www.gsi.ie/images/images/SAR_mapping_land_water.jpg)* </figcaption>
</figure>

Discover the items of Sentinel-1 microwave backscatter ($\sigma_0$ [1]) at a 20 meter resolution 

In [54]:
# Search the specific collection at the edoc catalog
search = eodc_catalog.search(
    collections="SENTINEL1_SIG0_20M",
    bbox=bounding_box,
    datetime=time_range,
)

# Get the items from this collection
items_sig0 = search.item_collection()

Helper functions are defined to extract specific information from the items

In [55]:
# Helper functions
# Functions  created to help extract some required information as:
# * Orbit state
# * Relative orbit number
# * scaling factor
# * nodata values


def extract_orbit_names(items: ItemCollection) -> np.ndarray:
    """Extract orbit state and relative orbit number from STAC Items."""
    return np.array(
        [
            items[i].properties["sat:orbit_state"][0].upper()
            + str(items[i].properties["sat:relative_orbit"])
            for i in range(len(items))
        ],
    )


def post_process_eodc_cube(
    dc: xr.Dataset,
    items: ItemCollection,
    bands: Sequence[str],
) -> xr.Dataset:
    """Apply post-processing to each dataset in the cube."""
    if not isinstance(bands, tuple):
        bands = tuple(bands)
    for band in bands:
        dc[band] = post_process_eodc_band(dc[band], items, band)
    return dc


def post_process_eodc_band(
    dc: xr.DataArray,
    items: ItemCollection,
    band: str,
) -> xr.DataArray:
    """Apply scaling-factor and no-data value to a single band."""
    scaling_factor = items[0].assets[band].extra_fields.get("raster:bands")[0]["scale"]
    no_data_value = items[0].assets[band].extra_fields.get("raster:bands")[0]["nodata"]
    return dc.where(dc != no_data_value) / scaling_factor


We now load the VV polarization of the discovered items with `odc-stac` with the previously defined projection and resolution. 

The data is, at this point, only lazily loaded (i.e., no actual data is read from disk).


In [56]:
bands = ["VV"]
sig0_dc = odc_stac.load(
    items_sig0,
    bands=bands,
    crs=crs,
    chunks=chunks,
    resolution=res,
    bbox=bounding_box,
    resampling="bilinear",
    groupby=None,
)

The data is then prepared for the next steps by: 
- filling the no data values with `np.nan` values
- extracting the orbit names, removing duplicated time dimension.

In [57]:
# Data preparation

sig0_dc = (
    post_process_eodc_cube(sig0_dc, items_sig0, bands)
    .rename_vars({"VV": "sig0"})
    .assign_coords(orbit=("time", extract_orbit_names(items_sig0)))
    .dropna(dim="time", how="all")
    .sortby("time")
)
_, indices = np.unique(sig0_dc.time, return_index=True)
indices.sort()
orbit_sig0 = sig0_dc.orbit[indices].data
sig0_dc = sig0_dc.groupby("time").mean(skipna=True)
sig0_dc = sig0_dc.assign_coords(orbit=("time", orbit_sig0))
sig0_dc = sig0_dc.persist()

### 3.2 Harmonic Parameters

The so-called likelihoods of $P(\sigma^0|flood)$ and $P(\sigma^0|nonflood)$ can be calculated from past backscattering information. 

To be able to do this, we load the harmonic parameters so we can model the expected variations in land back scattering based on seasonal changes in vegetation. The procedure is similar to the backscattering routine.



Discover Harmonic Parameters items.

In [58]:
# Search the specific collection at the EODC Catalog
search = eodc_catalog.search(collections="SENTINEL1_HPAR", bbox=bounding_box)

# Get the items
items_hpar = search.item_collection()

Load the data as a lazy object.


In [59]:
# Define the bands
bands = ("C1", "C2", "C3", "M0", "S1", "S2", "S3", "STD") 
# a value of k = 3 (k sine and cosine components) has been found to be sufficient to represent processes which occur on a time scale of ca. four months (Schlaffer et al. 2015)
# to practically derive the harmonic parameters (including the mean and standard deviation terms), we would need to select a specific time window from our time series and perform a least squares estimation analysis.
# The harmonic parameters are already produced offline for 2019-2021 (3 years).

# Load the data
hpar_dc = odc_stac.load(
    items_hpar,
    bands=bands,
    crs=crs,
    chunks=chunks,
    resolution=res,
    bbox=bounding_box,
    groupby=None,
)

# Data preparation
hpar_dc = post_process_eodc_cube(hpar_dc, items_hpar, bands).rename({"time": "orbit"})
hpar_dc["orbit"] = extract_orbit_names(items_hpar)
hpar_dc = hpar_dc.groupby("orbit").mean(skipna=True)
hpar_dc = hpar_dc.sel(orbit=orbit_sig0)
hpar_dc = hpar_dc.persist()


### 3.3 Local Incidence Angles (LIA)

Local incidence angles of measured microwave backscattering are also important for calculating reference backscatter values, but now specifically for water bodies. The procedure is largely the same as for the harmonic parameters.


In [60]:
# Search the specific collection at the edoc catalog
search = eodc_catalog.search(collections="SENTINEL1_MPLIA", bbox=bounding_box)

# Get the items
items_plia = search.item_collection()

Load the lazy object and preprocess.


In [61]:
# Define the band
bands = ["MPLIA"]

# Load the data
plia_dc = odc_stac.load(
    items_plia,
    bands=bands,
    crs=crs,
    chunks=chunks,
    resolution=res,
    bbox=bounding_box,
    groupby=None,
)

# Data preparation
plia_dc = post_process_eodc_cube(plia_dc, items_plia, bands).rename({"time": "orbit"})
plia_dc["orbit"] = extract_orbit_names(items_plia)
plia_dc = plia_dc.groupby("orbit").mean(skipna=True)
plia_dc = plia_dc.sel(orbit=orbit_sig0)
plia_dc = plia_dc.persist()

### 3.4 GFM (Global Flood Monitoring) Water Mask

For flood mapping we are only interested in microwave backscattering over what used to be land, as such, we need a way to mask water bodies. For this we use the GFM Water Mask Cover data from the EODC catalog.


Similarly, we discover the required items and load the data.

In [62]:
# Search the specific collection at the EODC catalog
search_gfm = eodc_catalog.search(
    collections="GFM",
    bbox=bounding_box,
    datetime="2023-10-11T05:33:43.000000000",  # choose one time stamp
)

# Get the items
items_gfm = search_gfm.item_collection()

# Load the data
gfm_dc = (
    odc_stac.load(
        items_gfm,
        bbox=bounding_box,
        crs=crs,
        bands=["reference_water_mask"],
        resolution=res,
    )
    .squeeze("time")
    .drop_vars("time")
    .rename_vars({"reference_water_mask": "wcover"})
)

## 4. Fuse cube

At this point we have created 4 datacubes: Microwave backscatter, Harmonic Parameters, Local Incidence Angles and Water mask.

Now we need to fuse all four and filter HAND value of above zero. Additionally, we can drop the orbit coordinates, as well as time slices that contain no land backscattering data.


In [63]:
# Merge the 4 data cubes
flood_dc = xr.merge([sig0_dc, plia_dc, hpar_dc, gfm_dc])

# Replace 255 to NAN in the water mask
max_value = 255
flood_dc = flood_dc.where(flood_dc.wcover != max_value)

# Reset orbit index
flood_dc = flood_dc.reset_index("orbit", drop=True)

# Rename orbit
flood_dc = flood_dc.rename({"orbit": "time"})

# Drop empty elements
flood_dc = flood_dc.dropna(dim="time", how="all", subset=["sig0"])

/tmp/ipykernel_380/2610772739.py:2: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  flood_dc = xr.merge([sig0_dc, plia_dc, hpar_dc, gfm_dc])


Visualize the created cube

In [64]:
flood_dc

<xarray.Dataset> Size: 2GB
Dimensions:      (time: 7, latitude: 1668, longitude: 4445)
Coordinates:
  * latitude     (latitude) float64 13kB 54.6 54.6 54.6 54.6 ... 54.3 54.3 54.3
  * longitude    (longitude) float64 36kB 12.3 12.3 12.3 12.3 ... 13.1 13.1 13.1
    spatial_ref  int32 4B 4326
  * time         (time) datetime64[ns] 56B 2023-10-11T05:33:18 ... 2023-10-23...
Data variables:
    sig0         (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    MPLIA        (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    C1           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    C2           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    C3           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    M0           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    S1           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    S2           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    S3           (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    STD          (time, latitude, longitude) float32 208MB dask.array<chunksize=(1, 1300, 1300), meta=np.ndarray>
    wcover       (latitude, longitude) float32 30MB 1.0 1.0 1.0 ... nan nan nan

## 5. Likelihoods

Now we are ready to calculate the likelihoods of microwave backscattering given flooding (or non flooding).


### 5.1 Water

We start with water which is the simplest calculation, where the hard coded values are coefficients of a regression model fitted to global water backscattering values.  


In [65]:
def calc_water_likelihood(dc: xr.Dataset) -> xr.DataArray:
    """Calculate the likelihood of water based on MPLIA, with fixed coefficients (slope and intercept hardcoded from the linear regression model for water)."""
    return dc.MPLIA * -0.394181 + -4.142015 
# Two different backscatter distributions are considered. The flood (or more precisely, water) distribution is derived by collecting numerous ocean pixels across different incidence angles.
# A model is then built using 0.5° incidence angle bins where the relationship is represented by a simple linear regression.
# The key assumption is that this linear relationship holds globally, unlike the non-flood distribution, which is defined locally for each individual pixel.

# Applying the function to the datacube
flood_dc["wbsc"] = calc_water_likelihood(flood_dc)

### 5.2 Land

For land backscattering, we construct the harmonic model from the parameters contained in the fused data cube. 
By doing so, we obtain a reference land backscattering value to compare with the actual observed sigma naught values.

In [66]:
def harmonic_expected_backscatter(dc: xr.Dataset) -> xr.DataArray:
    """Compute the harmonic expected backscatter."""
    w: float = np.pi * 2 / 365

    t: xr.DataArray = dc.time.dt.dayofyear
    wt: xr.DataArray = w * t

    M0: xr.DataArray = dc.M0
    S1: xr.DataArray = dc.S1
    S2: xr.DataArray = dc.S2
    S3: xr.DataArray = dc.S3
    C1: xr.DataArray = dc.C1
    C2: xr.DataArray = dc.C2
    C3: xr.DataArray = dc.C3
    hm_c1: xr.DataArray = (M0 + S1 * np.sin(wt)) + (C1 * np.cos(wt))
    hm_c2: xr.DataArray = (hm_c1 + S2 * np.sin(2 * wt)) + C2 * np.cos(2 * wt)
    hm_c3: xr.DataArray = (hm_c2 + S3 * np.sin(3 * wt)) + C3 * np.cos(3 * wt)
    return hm_c3
# Two different backscatter distributions are considered: the non-flood distribution, which is derived from the harmonic parameter retrievals and represents the expected backscatter of a pixel at a given time of year.
# The underlying assumption is that flood events have a negligible influence on the modeling of these harmonic backscatter patterns, which is why a three-year time window is used.

flood_dc["hbsc"] = harmonic_expected_backscatter(flood_dc)

## 6. Flood mapping

Having calculated the likelihoods, we can now move on to calculate the probability of flood given a pixel's $\sigma^0$. 
For that we use Bayesian statistics, where we first assume the values of the flood/not flood probability as 50%/50%, those are called *priors*. Afterwards we update those using the likelihood we just calculated to update those values, which are now called *posteriors*.







The following code block shows how we calculate the priors which allow use to predict whether it is likely if a land pixel became flooded.


In [67]:
def bayesian_flood_decision(
    dc: xr.Dataset,
) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    """Compute the Bayesian decision for flood mapping.

    Returns the not-flood posterior probability, flood posterior probability,
    and the decision.
    """
    nf_std: float = 2.754041 # STD of the water distribution (always the same for each PLIA value).
    sig0: xr.DataArray = dc.sig0
    std: xr.DataArray = dc.STD
    wbsc: xr.DataArray = dc.wbsc
    hbsc: xr.DataArray = dc.hbsc

    f_prob: xr.DataArray = (1.0 / (std * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * (((sig0 - wbsc) / nf_std) ** 2)
    )
    nf_prob: xr.DataArray = (1.0 / (nf_std * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * (((sig0 - hbsc) / nf_std) ** 2)
    )

    evidence: xr.DataArray = (nf_prob * 0.5) + (f_prob * 0.5)
    nf_post_prob: xr.DataArray = (nf_prob * 0.5) / evidence
    f_post_prob: xr.DataArray = (f_prob * 0.5) / evidence
    decision: xr.DataArray = xr.where(
        np.isnan(f_post_prob) | np.isnan(nf_post_prob),
        np.nan,
        np.greater(f_post_prob, nf_post_prob),
    )
    return nf_post_prob, f_post_prob, decision


flood_dc[["nf_post_prob", "flood_post_prob", "decision"]] = bayesian_flood_decision(
    flood_dc,
)

## 7. Postprocessing

We continue by improving our flood map by filtering out observations that we expect to have low sensitivity to flooding based on a predefined set of criteria.

These criteria include:
* Incidence Angles exceeding certain thresholds (will be masked out)
* Conflicting Distributions Identification
* Measurement Outliers Removal
* High Uncertainty Denial on Decision



In [68]:
def post_processing(dc: xr.Dataset) -> xr.DataArray:
    """Post-process the flood prediction data.

    Masking of:
    - Invalid MPLIA values
    - Invalid HBSC values
    - Measurement outliers
    - Low-certainty flood predictions

    Returns the flood decision array.
    """
    min_plia = 27 # These are hardcoded PLIA values that define the acquisition condition boundaries of Sentinel-1.
    max_plia = 48 # If the PLIA is below 27 or above 48, the pixel is considered to be in sloping terrain. Under such conditions, floods cannot occur.
    min_certainty = 0.8 
    # Flood decisions are generally sharp, but to ensure reliability we apply an uncertainty threshold. 
    # Conditional error (the probability of the losing class) ranges from 0 (perfect) to 0.5 (random). Decisions with error > 0.2 are discarded and marked as “no data.”
    
    # Mask out invalid MPLIA values
    dc: xr.Dataset = dc * np.logical_and(min_plia <= dc.MPLIA, max_plia >= dc.MPLIA)
    # Mask out invalid HBSC values
    # If the non-flood backscatter is lower than, or overlaps with, the flood backscatter, the pixel is masked. Here, hbsc denotes the mean of the non-flood distribution and wbsc the mean of the flood distribution.
    # 2.754041 is the constant STD of the flood distribution for each PLIA. This approach typically masks very low-backscatter areas. A threshold of 0.5 was chosen empirically, based on observations across many flood examples.
    dc: xr.Dataset = dc * (dc.hbsc > (dc.wbsc + 0.5 * 2.754041)) 
    land_bsc_lower: xr.DataArray = dc.hbsc - 3 * dc.STD
    land_bsc_upper: xr.DataArray = dc.hbsc + 3 * dc.STD
    water_bsc_upper: xr.DataArray = dc.wbsc + 3 * 2.754041
    mask_land_outliers: xr.DataArray = np.logical_and(
        land_bsc_lower < dc.sig0,
        dc.sig0 < land_bsc_upper,
    )
    mask_water_outliers: xr.DataArray = dc.sig0 < water_bsc_upper
    dc: xr.Dataset = dc * (mask_land_outliers | mask_water_outliers) # We mask pixels where the flood and non-flood distributions are entirely non-overlapping, i.e., extreme observations not represented by either distribution.
    # Mask out low-certainty flood predictions
    return (dc * (dc.flood_post_prob > min_certainty)).decision


flood_output = post_processing(flood_dc)

### 7.1 Removal of Speckles

Speckles are areas of one or a few pixels, which are likely the result of the diversity of scattering surfaces at a sub-pixel level. In this approach it is argued that small, solitary flood surfaces are unlikely. Hence speckles are removed by applying a smoothing filter which consists of a rolling window median along the x and y-axis simultaneously.


In [69]:
flood_output = (
    flood_output.rolling({"longitude": 5, "latitude": 5}, center=True)
    .median(skipna=True)
    .persist()
)


## 8. Results

In the following graphic we superimpose the data on a map and we can move the slider to see which areas become flooded over time. Please be aware that the images will take a bit longer to load when moving the slider for the first time.

In [ ]:
flood_output.hvplot.image(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    tiles=True,
    project=True,
    cmap=["rgba(0, 0, 1, 0.1)", "darkred"],
    cticks=[(0, "non-flood"), (1, "flood")],
    frame_height=400,
)

In [ ]:
# Shutdown the cluster
# cluster.close(shutdown=True)

In [ ]:
# for c in gateway.list_clusters():
#     gateway.connect(c.name).shutdown()